# 03 · EyePACS pretraining → APTOS (A1, the main model)

Pretrains on all 88k EyePACS images (re-split by patient), fine-tunes on APTOS, evaluates exactly like A0 on the same splits, and saves the tags `eyepacs-pretrain` and `a1`.

About 6 hours: run it with **Save Version → Save & Run All (Commit)** so it continues with the browser closed.

**Kaggle settings:** Accelerator *GPU T4 x2* (the code uses one GPU), Internet *on*,
secret `HF_TOKEN` (a Hugging Face token with write access) switched on for this notebook.
**Inputs:** APTOS 2019 Blindness Detection (competition); IDRiD (`aaryapatel98/indian-diabetic-retinopathy-image-dataset`); EyePACS 512 (`tantai31124/eyepacs-512`).

`%%bash` must be the first line of its cell; put notes in markdown cells, not above it.

In [ ]:
import os, json
os.chdir('/kaggle/working')
if not os.path.exists('retina-screen'):
    !git clone -q https://github.com/ashwinruke/retina-screen.git
%cd /kaggle/working/retina-screen
!git pull -q
!pip install -q timm albumentations onnx onnxscript onnxruntime
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!nvidia-smi --query-gpu=name --format=csv,noheader

## Splits
Restores A0's APTOS and IDRiD splits (so A0 and A1 are tested on identical images) and builds the EyePACS manifest from its grade folders.

In [ ]:
!python scripts/artifacts.py pull --tag mvp
!python scripts/prepare_data.py --eyepacs /kaggle/input/datasets/tantai31124/eyepacs-512/EYEPACS_512_final --out-dir data/manifests

## Pipeline (~6 h)
The EyePACS cache (~8 GB) goes in `/tmp` so it isn't saved with the notebook output. The pretrained model is saved before fine-tuning starts.

In [ ]:
%%bash
set -e
cd /kaggle/working/retina-screen
python scripts/cache_images.py --manifest data/manifests/eyepacs.csv --out-dir /tmp/cache --config configs/pretrain_eyepacs.yaml
python scripts/cache_images.py --manifest data/manifests/aptos.csv --out-dir cache --config configs/finetune_aptos.yaml
python scripts/cache_images.py --manifest data/manifests/idrid.csv --out-dir cache --config configs/finetune_aptos.yaml

python scripts/train.py --config configs/pretrain_eyepacs.yaml data.manifest=/tmp/cache/eyepacs.csv
python scripts/artifacts.py push --tag eyepacs-pretrain --run pretrain_eyepacs --include-last --skip-model

python scripts/train.py --config configs/finetune_aptos.yaml
python scripts/evaluate.py --checkpoint outputs/finetune_aptos/best.pt \
    --manifest cache/aptos.csv --splits test \
    --fit-manifest cache/aptos.csv --fit-splits val \
    --name A1_aptos_test --export outputs/model.pt
python scripts/evaluate.py --checkpoint outputs/model.pt --manifest cache/idrid.csv --splits all  --name A1_idrid
python scripts/evaluate.py --checkpoint outputs/model.pt --manifest cache/idrid.csv --splits test --name A1_idrid_test

python scripts/plot_results.py --history outputs/finetune_aptos/history.csv \
    --report outputs/eval/A1_aptos_test.json \
    --predictions outputs/eval/A1_aptos_test_predictions.csv --out-dir docs/figures --prefix a1
python scripts/plot_results.py --report outputs/eval/A1_idrid.json \
    --predictions outputs/eval/A1_idrid_predictions.csv --out-dir docs/figures --prefix a1_idrid

python scripts/artifacts.py push --tag a1 --run pretrain_eyepacs finetune_aptos

In [ ]:
def show(*names):
    for name in names:
        r = json.load(open(f'outputs/eval/{name}.json'))['tuned_thresholds']
        rec = r['per_grade_recall']
        print(f"{name:20s} n={r['n']:4d} QWK={r['qwk']:.3f} AUC={r['referable']['auc']:.3f} "
              f"sens={r['referable']['sensitivity']:.3f} spec={r['referable']['specificity']:.3f} "
              f"severe={rec['Severe NPDR']:.2f} prolif={rec['Proliferative DR']:.2f} "
              f"cuts={[round(t, 2) for t in r['thresholds']]}")
show('mvp_aptos_test', 'A1_aptos_test', 'A0_idrid', 'A1_idrid', 'A0_idrid_test', 'A1_idrid_test')
print(open('outputs/pretrain_eyepacs/history.csv').read())

## Deploy A1 to the demo
Run this once you've checked the results (e.g. in a 02 session after `pull --tag a1`).

In [ ]:
!python scripts/export_onnx.py --model outputs/model.pt --out-dir web \
    --check-images data/manifests/aptos.csv --report outputs/eval/A1_aptos_test.json \
    --train-data "EyePACS 2015 (pretraining) and APTOS 2019 fundus photos"
!python scripts/deploy_space.py --space ashwinruke/retina-dr-screening